simple csv file

In [14]:
import pandas as pd

# =========================
# EXTRACT
# =========================
def extract(file_path):
    print("Extracting data...")
    df = pd.read_csv(file_path)
    return df


# =========================
# TRANSFORM
# =========================
def transform(df):
    print("Transforming data...")

    # Example transformations:

    # 1. Drop duplicates
    df = df.drop_duplicates()

    # 2. Handle missing values
    df = df.fillna({
        'Age': df['Age'].mean(),      # fill numeric
        'Name': 'Unknown'             # fill categorical
    })

    # 3. Create new column
    if 'Salary' in df.columns:
        df['Salary_After_Tax'] = df['Salary'] * 0.8

    # 4. Rename columns
    df.columns = [col.strip().lower() for col in df.columns]

    return df


# =========================
# LOAD
# =========================
def load(df, output_path):
    print("Loading data...")
    df.to_csv(output_path, index=False)
    print(f"Data saved to {output_path}")


# =========================
# MAIN PIPELINE
# =========================
def etl_pipeline(input_file, output_file):
    df = extract(input_file)
    df = transform(df)
    load(df, output_file)


# =========================
# RUN
# =========================
if __name__ == "__main__":
    input_file = "input_data.csv"
    output_file = "processed_data.csv"

    etl_pipeline(input_file, output_file)

Extracting data...
Transforming data...
Loading data...
Data saved to processed_data.csv


multiple tables

In [11]:
import pandas as pd
import sqlite3
from datetime import datetime

# =========================
# EXTRACT
# =========================
def extract(file_path):
    return pd.read_csv(file_path)


# =========================
# TRANSFORM
# =========================
def transform(df):
    df = df.drop_duplicates().copy()

    # Fill missing
    df["Age"] = df["Age"].fillna(df["Age"].mean())

    # Add department (example logic)
    df["Department"] = ["IT", "HR", "IT", "Finance"][:len(df)]

    # Salary after tax
    df["Salary_After_Tax"] = df["Salary"] * 0.8

    # Add timestamp
    df["Date"] = datetime.now().strftime("%Y-%m-%d")

    return df


# =========================
# LOAD (Normalized DB)
# =========================
def load(df, db_name="advanced_etl.db"):
    conn = sqlite3.connect(db_name)
    cursor = conn.cursor()

    # Create tables
    cursor.executescript("""
    CREATE TABLE IF NOT EXISTS departments (
        department_id INTEGER PRIMARY KEY AUTOINCREMENT,
        department_name TEXT UNIQUE
    );

    CREATE TABLE IF NOT EXISTS employees (
        employee_id INTEGER PRIMARY KEY AUTOINCREMENT,
        name TEXT,
        age REAL,
        department_id INTEGER,
        FOREIGN KEY (department_id) REFERENCES departments(department_id)
    );

    CREATE TABLE IF NOT EXISTS salaries (
        salary_id INTEGER PRIMARY KEY AUTOINCREMENT,
        employee_id INTEGER,
        salary REAL,
        salary_after_tax REAL,
        date TEXT,
        FOREIGN KEY (employee_id) REFERENCES employees(employee_id)
    );
    """)

    # Insert departments
    for dept in df["Department"].unique():
        cursor.execute(
            "INSERT OR IGNORE INTO departments (department_name) VALUES (?)",
            (dept,)
        )

    # Insert employees
    for _, row in df.iterrows():
        cursor.execute("SELECT department_id FROM departments WHERE department_name = ?", (row["Department"],))
        dept_id = cursor.fetchone()[0]

        cursor.execute("""
            INSERT INTO employees (name, age, department_id)
            VALUES (?, ?, ?)
        """, (row["Name"], row["Age"], dept_id))

        emp_id = cursor.lastrowid

        # Insert salary
        cursor.execute("""
            INSERT INTO salaries (employee_id, salary, salary_after_tax, date)
            VALUES (?, ?, ?, ?)
        """, (emp_id, row["Salary"], row["Salary_After_Tax"], row["Date"]))

    conn.commit()
    conn.close()


# =========================
# RUN
# =========================
if __name__ == "__main__":
    df = extract("input_data.csv")
    df = transform(df)
    load(df)

using database

In [15]:
import pandas as pd
import sqlite3

# =========================
# EXTRACT
# =========================
def extract(file_path):
    print("Extracting data...")
    df = pd.read_csv(file_path)
    return df


# =========================
# TRANSFORM
# =========================
def transform(df):
    print("Transforming data...")

    # Remove duplicates
    df = df.drop_duplicates()

    # Handle missing values
    if 'Age' in df.columns:
        df['Age'] = df['Age'].fillna(df['Age'].mean())

    if 'Name' in df.columns:
        df['Name'] = df['Name'].fillna('Unknown')

    # Feature engineering
    if 'Salary' in df.columns:
        df['Salary_After_Tax'] = df['Salary'] * 0.8

    # Normalize column names
    df.columns = [col.strip().lower() for col in df.columns]

    return df


# =========================
# LOAD (SQL)
# =========================
def load(df, db_name, table_name):
    print("Loading data into SQL database...")

    conn = sqlite3.connect(db_name)
    cursor = conn.cursor()

    # Create table if not exists
    create_table_query = f"""
    CREATE TABLE IF NOT EXISTS {table_name} (
        name TEXT,
        age REAL,
        salary REAL,
        salary_after_tax REAL
    )
    """
    cursor.execute(create_table_query)

    # Insert data
    df.to_sql(table_name, conn, if_exists='replace', index=False)

    conn.commit()
    conn.close()

    print(f"Data loaded into table '{table_name}' in database '{db_name}'")


# =========================
# MAIN PIPELINE
# =========================
def etl_pipeline(input_file, db_name, table_name):
    df = extract(input_file)
    df = transform(df)
    load(df, db_name, table_name)


# =========================
# RUN
# =========================
if __name__ == "__main__":
    input_file = "input_data.csv"
    db_name = "etl_database.db"
    table_name = "employees"

    etl_pipeline(input_file, db_name, table_name)

Extracting data...
Transforming data...
Loading data into SQL database...
Data loaded into table 'employees' in database 'etl_database.db'


/tmp/ipykernel_6119/2884651566.py:24: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['Age'] = df['Age'].fillna(df['Age'].mean())
/tmp/ipykernel_6119/2884651566.py:27: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['Name'] = df['Name'].fillna('Unknown')
/tmp/ipykernel_6119/2884651566.py:31: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pand

real example

In [16]:
import pandas as pd
import sqlite3
import logging
from datetime import datetime
import numpy as np  # Added for np.sqrt

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.ensemble import RandomForestRegressor

# =========================
# CONFIG
# =========================
RAW_FILE = "input_data.csv"
STAGING_DB = "staging.db"
WAREHOUSE_DB = "warehouse.db"

logging.basicConfig(
    filename="etl_ml.log",
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s"
)

# =========================
# EXTRACT
# =========================
def extract():
    df = pd.read_csv(RAW_FILE)
    return df

# =========================
# VALIDATE
# =========================
def validate(df):
    if "Salary" not in df.columns:
        raise ValueError("Missing Salary column")
    return df

# =========================
# TRANSFORM
# =========================
def transform(df):
    df = df.drop_duplicates().copy()

    df["Age"] = df["Age"].fillna(df["Age"].mean())
    df["Salary_After_Tax"] = df["Salary"] * 0.8

    df["Department"] = ["IT", "HR", "Finance", "IT"][:len(df)]
    df["Load_Timestamp"] = datetime.now()

    return df

# =========================
# LOAD → STAGING
# =========================
def load_staging(df):
    conn = sqlite3.connect(STAGING_DB)
    df.to_sql("staging_employees", conn, if_exists="replace", index=False)
    conn.close()

# =========================
# LOAD → WAREHOUSE
# =========================
def load_warehouse():
    staging_conn = sqlite3.connect(STAGING_DB)
    wh_conn = sqlite3.connect(WAREHOUSE_DB)

    df = pd.read_sql("SELECT * FROM staging_employees", staging_conn)
    cursor = wh_conn.cursor()

    cursor.executescript("""
    CREATE TABLE IF NOT EXISTS departments (
        department_id INTEGER PRIMARY KEY AUTOINCREMENT,
        department_name TEXT UNIQUE
    );

    CREATE TABLE IF NOT EXISTS employees (
        employee_id INTEGER PRIMARY KEY AUTOINCREMENT,
        name TEXT,
        age REAL,
        department_id INTEGER,
        load_timestamp TEXT
    );

    CREATE TABLE IF NOT EXISTS salaries (
        salary_id INTEGER PRIMARY KEY AUTOINCREMENT,
        employee_id INTEGER,
        salary REAL,
        salary_after_tax REAL
    );
    """)

    for _, row in df.iterrows():
        cursor.execute(
            "INSERT OR IGNORE INTO departments (department_name) VALUES (?)",
            (row["Department"],)
        )

        cursor.execute(
            "SELECT department_id FROM departments WHERE department_name=?",
            (row["Department"],)
        )
        dept_id = cursor.fetchone()[0]

        cursor.execute("""
            INSERT INTO employees (name, age, department_id, load_timestamp)
            VALUES (?, ?, ?, ?)
        """, (row["Name"], row["Age"], dept_id, str(row["Load_Timestamp"])))

        emp_id = cursor.lastrowid

        cursor.execute("""
            INSERT INTO salaries (employee_id, salary, salary_after_tax)
            VALUES (?, ?, ?)
        """, (emp_id, row["Salary"], row["Salary_After_Tax"]))

    wh_conn.commit()
    staging_conn.close()
    wh_conn.close()

# =========================
# ML PIPELINE
# =========================
def run_ml_pipeline():
    conn = sqlite3.connect(WAREHOUSE_DB)

    query = """
    SELECT e.age, d.department_name, s.salary
    FROM employees e
    JOIN departments d ON e.department_id = d.department_id
    JOIN salaries s ON e.employee_id = s.employee_id
    """

    df = pd.read_sql(query, conn)
    conn.close()

    # Encode categorical
    df = pd.get_dummies(df, columns=["department_name"], drop_first=True)

    X = df.drop("salary", axis=1)
    y = df["salary"]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )

    model = RandomForestRegressor()
    model.fit(X_train, y_train)

    predictions = model.predict(X_test)

    rmse = np.sqrt(mean_squared_error(y_test, predictions))
    r2 = r2_score(y_test, predictions)

    print("RMSE:", rmse)
    print("R2 Score:", r2)

    logging.info(f"RMSE: {rmse}, R2: {r2}")

    return model

# =========================
# MAIN PIPELINE
# =========================
def run_pipeline():
    df = extract()
    df = validate(df)
    df = transform(df)

    load_staging(df)
    load_warehouse()

    model = run_ml_pipeline()

    return model


# =========================
# RUN
# =========================
if __name__ == "__main__":
    run_pipeline()

RMSE: 85.82928793055822
R2 Score: 0.98895
